# Template 05: Model Training

**Purpose:** Train XGBoost model and generate predictions

**Inputs:**
- data/04_train.parquet
- data/04_test.parquet

**Outputs:**
- models/xgb_model.json
- results/05_metrics.yaml
- results/05_predictions.parquet

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for papermill

import pandas as pd
import numpy as np
import xgboost as xgb
import yaml
import os
import sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 05: MODEL TRAINING")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
target = cfg['experiment']['target']

In [ ]:
# Load train/test data
train = pd.read_parquet(f"{output_base}/data/04_train.parquet")
test = pd.read_parquet(f"{output_base}/data/04_test.parquet")

print(f"\n* Train: {train.shape}")
print(f"* Test: {test.shape}")

In [ ]:
# Load feature list
features_df = pd.read_csv(f"{config_path}/{cfg['features']['inclusion_file']}", comment='#')
feature_cols = features_df['column_name'].tolist()

# Filter to available features
available_features = [f for f in feature_cols if f in train.columns]
print(f"\n* Features: {len(available_features)}/{len(feature_cols)} available")

X_train = train[available_features]
y_train = train[target]
X_test = test[available_features]
y_test = test[target]

In [ ]:
# Train XGBoost model
print(f"\n* Training XGBoost...")

xgb_params = cfg['xgboost'].copy()
n_estimators = xgb_params.pop('n_estimators', 5000)

model = xgb.XGBRegressor(n_estimators=n_estimators, **xgb_params)
model.fit(X_train, y_train, verbose=100)

print(f"\n* Model trained")

In [ ]:
# Predictions
train['pred'] = model.predict(X_train)
test['pred'] = model.predict(X_test)

# Metrics
train_mae = mean_absolute_error(y_train, train['pred'])
test_mae = mean_absolute_error(y_test, test['pred'])
train_rmse = np.sqrt(mean_squared_error(y_train, train['pred']))
test_rmse = np.sqrt(mean_squared_error(y_test, test['pred']))

print(f"\n* Metrics:")
print(f"  Train MAE: {train_mae:.4f}")
print(f"  Test MAE: {test_mae:.4f}")
print(f"  Train RMSE: {train_rmse:.4f}")
print(f"  Test RMSE: {test_rmse:.4f}")

In [ ]:
# Save model
model_file = f"{output_base}/models/xgb_model.json"
os.makedirs(f"{output_base}/models", exist_ok=True)
model.save_model(model_file)
print(f"\n* Model saved: {model_file}")

In [ ]:
# Save metrics
metrics = {
    'train_mae': float(train_mae),
    'test_mae': float(test_mae),
    'train_rmse': float(train_rmse),
    'test_rmse': float(test_rmse),
    'n_features': len(available_features),
    'train_size': len(train),
    'test_size': len(test)
}

metrics_file = f"{output_base}/results/05_metrics.yaml"
with open(metrics_file, 'w') as f:
    yaml.dump(metrics, f)

print(f"* Metrics saved: {metrics_file}")

In [ ]:
# Save predictions
pred_file = f"{output_base}/results/05_predictions.parquet"
test[['pred', target]].to_parquet(pred_file)
print(f"* Predictions saved: {pred_file}")

## Lift Charts

In [ ]:
# Prepare data for lift charts
# Get exposure column from config, or use fallback logic
exposure_col = cfg['experiment'].get('exposure', None)

if exposure_col and exposure_col in train.columns:
    weight_col = exposure_col
    print(f"\n* Using '{exposure_col}' from config as weight column")
elif 'ee' in train.columns:
    weight_col = 'ee'
    print(f"\n* Using 'ee' (earned exposure) as weight column")
elif 'ee_imps' in train.columns:
    weight_col = 'ee_imps'
    print(f"\n* Using 'ee_imps' (earned exposure imputed) as weight column")
else:
    train['weight'] = 1  # Equal weight per record
    test['weight'] = 1
    weight_col = 'weight'
    print(f"\n* WARNING: No exposure column found, using equal weighting (weight=1)")
    if exposure_col:
        print(f"  Config specifies '{exposure_col}' but it's not in the data!")

# Create incurred/denom columns for lift chart
# Use exposure as denominator if available
if weight_col in train.columns and weight_col != 'weight':
    train['incurred_act'] = train[target]
    train['incurred_pred'] = train['pred']
    train['denom'] = train[weight_col]  # Use exposure as denominator
    
    test['incurred_act'] = test[target]
    test['incurred_pred'] = test['pred']
    test['denom'] = test[weight_col]
else:
    train['incurred_act'] = train[target]
    train['incurred_pred'] = train['pred']
    train['denom'] = 1  # Equal weight
    
    test['incurred_act'] = test[target]
    test['incurred_pred'] = test['pred']
    test['denom'] = 1

print(f"* Prepared data for lift charts (weight_col='{weight_col}')")

In [ ]:
# Load optimized lift chart function
import matplotlib.pyplot as plt
from lift_chart_fast import create_lift_chart

print("* Optimized lift chart function loaded")

In [ ]:
print("\n* Generating train lift chart...")


In [ ]:
fig_train, table_train = create_lift_chart(train, weight_col, bins=10, title="Train Lift Chart")

In [ ]:
# Train lift chart


train_chart_file = f"{output_base}/results/05_lift_chart_train.png"
fig_train.savefig(train_chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_train)

print(f"  Saved: {train_chart_file}")

# Display inline in notebook
from IPython.display import Image, display
display(Image(train_chart_file))

print("\nTrain decile table:")
print(table_train[['decile', 'act', 'pred', 'act_rel', 'pred_rel']])

In [ ]:
# Test lift chart
print("\n* Generating test lift chart...")
fig_test, table_test = create_lift_chart(test, weight_col, bins=10, title="Test Lift Chart")

test_chart_file = f"{output_base}/results/05_lift_chart_test.png"
fig_test.savefig(test_chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_test)

print(f"  Saved: {test_chart_file}")

# Display inline in notebook
display(Image(test_chart_file))

print("\nTest decile table:")
print(table_test[['decile', 'act', 'pred', 'act_rel', 'pred_rel']])

In [ ]:
print("\n########################################")
print("# STAGE 05: COMPLETE")
print("########################################")